# E-Commerce Orders Pipeline Analysis

This notebook analyzes all tables in the `main.demo_sdp` pipeline schema, explaining the medallion architecture (Bronze → Silver → Gold) and data quality flow.

## Pipeline Summary
- **Source:** JSON files from `/Volumes/main/default/demo_orders`
- **Architecture:** Bronze (raw) → Quarantine (failures) → Silver (validated) → Gold (analytics)
- **Total Records:** 1,000 raw → 919 validated → aggregated metrics

## 📥 BRONZE LAYER: Raw Data Ingestion

**Purpose:** Capture ALL data from source files without any filtering or transformation.

**Table:** `bronze_orders` (STREAMING_TABLE)
- **Records:** 1,000 total
- **Source:** Auto Loader reading JSON files from Volume
- **Key Feature:** Preserves everything - good, bad, and ugly data
- **Metadata Added:** `ingestion_time`, `source_file`, `file_modified_time`, `operation_type`

**Why it exists:** Provides complete audit trail and enables reprocessing if needed.

In [0]:
%sql
-- Sample of raw ingested orders
SELECT 
  order_id,
  user_id,
  product_id,
  amount,
  status,
  order_timestamp,
  source_file,
  operation_type,
  ingestion_time
FROM main.demo_sdp.bronze_orders
LIMIT 10;

In [0]:
%sql
-- Bronze layer statistics
SELECT 
  COUNT(*) as total_records,
  COUNT(DISTINCT source_file) as total_files,
  MIN(ingestion_time) as first_ingestion,
  MAX(ingestion_time) as last_ingestion
FROM main.demo_sdp.bronze_orders;

## 🚨 QUARANTINE LAYER: Data Quality Failures

**Purpose:** Isolate records that fail quality checks for investigation and repair.

**Table:** `quarantine_orders` (STREAMING_TABLE)
- **Records:** 81 failed (8.1% failure rate)
- **Quality Rules:**
  - `user_id IS NULL` → Cannot identify customer
  - `amount <= 0` → Invalid pricing data
  - `order_id IS NULL` → Missing unique identifier

**Why records go here:** Bad data that would corrupt downstream analytics gets quarantined instead of dropped, allowing:
1. Root cause analysis
2. Data source fixes
3. Potential recovery/correction
4. Quality monitoring

**How it works:** Parallel stream from bronze captures violations BEFORE silver layer drops them.

In [0]:
%sql
-- Why did 81 records fail quality checks?
SELECT 
  quarantine_reason,
  COUNT(*) as failed_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM main.demo_sdp.quarantine_orders), 2) as pct_of_failures
FROM main.demo_sdp.quarantine_orders
GROUP BY quarantine_reason
ORDER BY failed_count DESC;

In [0]:
%sql
-- Examples of quarantined records
SELECT 
  order_id,
  user_id,
  product_id,
  amount,
  quarantine_reason,
  quarantine_time,
  source_file
FROM main.demo_sdp.quarantine_orders
LIMIT 5;

## ✅ SILVER LAYER: Validated & Cleaned Data

**Purpose:** Provide clean, validated data ready for analytics.

### Table 1: `silver_orders` (STREAMING_TABLE)
- **Records:** 919 validated (1,000 - 81 quarantined)
- **Quality Constraints:**
  - `valid_user_id`: user_id IS NOT NULL → DROP ROW on violation
  - `valid_amount`: amount > 0 → DROP ROW on violation
  - `valid_order_id`: order_id IS NOT NULL → DROP ROW on violation
- **Enrichments:**
  - Converted `order_timestamp` from string to TIMESTAMP
  - Added `processed_time` timestamp

### Table 2: `silver_orders_deduped` (STREAMING_TABLE)
- **Purpose:** Remove duplicate orders using deduplication logic
- **Configuration:**
  - Deduplication using ROW_NUMBER() partitioned by `order_id`
  - Latest record selected by `processed_time DESC`
- **Additional Column:**
  - `is_late_data`: Flag indicating if data arrived >1 hour after file modification time
- **How it works:** If same order_id appears multiple times, only the record with the latest processed_time is kept

**Why two silver tables:** 
1. `silver_orders` = all validated records (may have duplicates)
2. `silver_orders_deduped` = deduplicated with late data tracking (one record per order_id)

In [0]:
%sql
-- Validated orders that passed all quality checks
SELECT 
  order_id,
  user_id,
  product_id,
  amount,
  status,
  order_timestamp,
  is_late_data,
  processed_time
FROM main.demo_sdp.silver_orders_deduped
LIMIT 10;

In [0]:
%sql
-- How many duplicates were removed by Auto CDC?
SELECT 
  'Before CDC (silver_orders)' as stage,
  COUNT(*) as total_records,
  COUNT(DISTINCT order_id) as unique_orders,
  COUNT(*) - COUNT(DISTINCT order_id) as duplicates
FROM main.demo_sdp.silver_orders

UNION ALL

SELECT 
  'After CDC (silver_orders_deduped)' as stage,
  COUNT(*) as total_records,
  COUNT(DISTINCT order_id) as unique_orders,
  0 as duplicates
FROM main.demo_sdp.silver_orders_deduped;

In [0]:
%sql
-- How much data arrived late (>24 hours after order timestamp)?
SELECT 
  is_late_data,
  COUNT(*) as record_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM main.demo_sdp.silver_orders_deduped), 2) as percentage
FROM main.demo_sdp.silver_orders_deduped
GROUP BY is_late_data
ORDER BY is_late_data;

## 🏆 GOLD LAYER: Business Analytics

**Purpose:** Aggregated, business-ready metrics for dashboards and reports.

**Why MATERIALIZED VIEWS:** Gold tables use aggregations (SUM, COUNT, AVG) that need to recompute when source data changes. Materialized Views automatically refresh when upstream data updates.

### Tables:
1. **`gold_user_dimension`** - User lifetime metrics and segmentation
2. **`gold_product_performance`** - Product sales metrics
3. **`gold_hourly_metrics`** - Time-based aggregations
4. **`gold_daily_user_sales`** - Daily user purchasing patterns

### Gold Table 1: User Dimension

**Aggregates:** MIN, MAX, COUNT, SUM, AVG per user_id

**Metrics:**
- `lifetime_orders`: Total orders per user
- `lifetime_value`: Total spend (SUM of amounts)
- `avg_order_value`: Average order size
- `first_order_date` / `last_order_date`: Customer tenure

**Segmentation:**
- **Customer Tier:** VIP (>$10K), GOLD ($5K-$10K), SILVER ($1K-$5K), BRONZE (<$1K)
- **Customer Status:** Active (ordered in last 30 days), At Risk (31-90 days), Churned (>90 days)

**Why it exists:** Powers customer segmentation, lifetime value analysis, and retention metrics.

In [0]:
%sql
-- How many customers in each tier?
SELECT 
  customer_tier,
  COUNT(*) as customer_count,
  ROUND(AVG(lifetime_value), 2) as avg_lifetime_value,
  ROUND(MIN(lifetime_value), 2) as min_value,
  ROUND(MAX(lifetime_value), 2) as max_value
FROM main.demo_sdp.gold_user_dimension
GROUP BY customer_tier
ORDER BY 
  CASE customer_tier
    WHEN 'VIP' THEN 1
    WHEN 'GOLD' THEN 2
    WHEN 'SILVER' THEN 3
    WHEN 'BRONZE' THEN 4
  END;

In [0]:
%sql
-- Top 10 customers by lifetime value
SELECT 
  user_id,
  customer_tier,
  customer_status,
  lifetime_orders,
  ROUND(lifetime_value, 2) as lifetime_value,
  ROUND(avg_order_value, 2) as avg_order_value,
  DATE(first_order_date) as first_order,
  DATE(last_order_date) as last_order
FROM main.demo_sdp.gold_user_dimension
ORDER BY lifetime_value DESC
LIMIT 10;

In [0]:
%sql
-- Customer status: Active vs At Risk vs Churned
SELECT 
  customer_status,
  COUNT(*) as customer_count,
  ROUND(AVG(lifetime_value), 2) as avg_lifetime_value
FROM main.demo_sdp.gold_user_dimension
GROUP BY customer_status
ORDER BY customer_count DESC;

### Gold Table 2: Product Performance

**Aggregates:** COUNT, SUM, AVG per product_id

**Metrics:**
- `total_orders`: How many times product was ordered
- `total_revenue`: Total sales (SUM of amounts)
- `avg_price`: Average order amount for this product
- `unique_customers`: Distinct users who bought it

**Why it exists:** Identifies best-selling products, revenue drivers, and customer reach per product.

In [0]:
%sql
-- Top 10 products by total revenue
SELECT 
  product_id,
  total_orders,
  ROUND(total_revenue, 2) as total_revenue,
  ROUND(avg_price, 2) as avg_price,
  unique_customers,
  ROUND(total_revenue / NULLIF(unique_customers, 0), 2) as revenue_per_customer
FROM main.demo_sdp.gold_product_performance
ORDER BY total_revenue DESC
LIMIT 10;

In [0]:
%sql
-- Products by order volume (most frequently ordered)
SELECT 
  product_id,
  total_orders,
  unique_customers,
  ROUND(total_revenue, 2) as total_revenue,
  ROUND(avg_price, 2) as avg_price
FROM main.demo_sdp.gold_product_performance
ORDER BY total_orders DESC
LIMIT 10;

### Gold Table 3: Hourly Metrics

**Aggregates:** Time-based metrics by date and hour

**Metrics:**
- `orders_count`, `total_revenue`, `avg_order_value`
- `active_users`: Distinct users per hour
- `products_sold`: Distinct products per hour
- `late_data_count` / `late_data_pct`: Late arrival tracking

**Why it exists:** Powers time-series dashboards, identifies peak hours, tracks data freshness.

In [0]:
%sql
-- Hourly processing metrics
SELECT 
  metric_date,
  metric_hour,
  orders_count,
  ROUND(total_revenue, 2) as total_revenue,
  ROUND(avg_order_value, 2) as avg_order_value,
  active_users,
  products_sold,
  late_data_count,
  late_data_pct
FROM main.demo_sdp.gold_hourly_metrics
ORDER BY metric_date DESC, metric_hour DESC;

### Gold Table 4: Daily User Sales

**Aggregates:** Per user per day metrics

**Metrics:**
- `total_orders`, `total_sales`, `avg_order_value` per user per day
- `unique_products`: Product variety purchased

**Why it exists:** Tracks daily user behavior, identifies high-value shopping days, enables cohort analysis.

In [0]:
%sql
-- Top 10 user-day combinations by sales
SELECT 
  order_date,
  user_id,
  total_orders,
  ROUND(total_sales, 2) as total_sales,
  ROUND(avg_order_value, 2) as avg_order_value,
  unique_products
FROM main.demo_sdp.gold_daily_user_sales
ORDER BY total_sales DESC
LIMIT 10;

In [0]:
%sql
-- Daily aggregated sales trend
SELECT 
  order_date,
  COUNT(DISTINCT user_id) as active_users,
  SUM(total_orders) as total_orders,
  ROUND(SUM(total_sales), 2) as total_sales,
  ROUND(AVG(avg_order_value), 2) as avg_order_value
FROM main.demo_sdp.gold_daily_user_sales
GROUP BY order_date
ORDER BY order_date DESC;

## 📊 OBSERVABILITY: Monitoring & Audit

**Purpose:** Track pipeline health, data quality, and CDC operations.

### Tables:
1. **`cdc_audit_log`** - Tracks all CDC change events (inserts/updates/deletes)
2. **`data_quality_metrics`** - Ingestion statistics by time
3. **`cdc_metrics_summary`** - Aggregated CDC operation counts
4. **`quarantine_summary`** - Aggregated quarantine reasons

**Why it exists:** Enables monitoring, alerting, and debugging of the pipeline.

In [0]:
%sql
-- Sample of CDC change events tracked
SELECT 
  order_id,
  user_id,
  operation_type,
  processing_date,
  processing_hour,
  ROUND(amount, 2) as amount
FROM main.demo_sdp.cdc_audit_log;

In [0]:
%sql
-- What CDC operations occurred?
SELECT 
  operation_type,
  COUNT(*) as event_count
FROM main.demo_sdp.cdc_audit_log
GROUP BY operation_type
ORDER BY event_count DESC;

In [0]:
%sql
-- Ingestion statistics
SELECT 
  metric_date,
  metric_hour,
  total_records_ingested,
  files_processed,
  first_record_time,
  last_record_time
FROM main.demo_sdp.data_quality_metrics
ORDER BY metric_date DESC, metric_hour DESC;

In [0]:
%sql
-- Aggregated quarantine failures
SELECT 
  quarantine_date,
  quarantine_reason,
  quarantine_count,
  affected_files
FROM main.demo_sdp.quarantine_summary
ORDER BY quarantine_count DESC;

## 📈 Complete Pipeline Flow Summary

```
1,000 Records (Bronze)
    |
    ├─→ 81 Records (Quarantine) ──→ Isolated for investigation
    │     ├─ 50: NULL user_id
    │     └─ 31: Invalid amount ≤ $0
    │
    └─→ 919 Records (Silver Validated)
          ├─ Quality constraints applied
          ├─ Enrichments added (is_late_data flag)
          └─→ CDC Deduplication (silver_orders_deduped)
                └─→ Gold Layer Aggregations
                      ├─ 227 Users (gold_user_dimension)
                      │   ├─ 4 GOLD ($5K-$10K)
                      │   ├─ 51 SILVER ($1K-$5K)
                      │   └─ 172 BRONZE (<$1K)
                      │
                      ├─ 193 Products (gold_product_performance)
                      ├─ 1 Hourly Record (gold_hourly_metrics)
                      └─ 504 User-Day Records (gold_daily_user_sales)
```

### Key Insights:
- **8.1% data quality failure rate** (81/1,000) was caught and quarantined
- **70% late data** arrived more than 24 hours after order timestamp
- **No duplicates** in this dataset (all order_ids were unique)
- **Top customer:** user_0046 with $5,900 lifetime value
- **Top product:** product_0061 with $3,478 revenue

### Why This Architecture Works:
1. **Bronze** captures raw data → audit trail + reprocessing capability
2. **Quarantine** isolates bad data → prevents analytics corruption
3. **Silver** validates & enriches → clean data foundation
4. **CDC** deduplicates → handles duplicates correctly
5. **Gold** aggregates → business-ready metrics
6. **Observability** monitors → pipeline health visibility

**Without quarantine layer:** Those 81 bad records would have been silently dropped or corrupted downstream metrics!